1. Loading and Preprocessing

Load the dataset and perform necessary preprocessing steps. This should include text
cleaning, tokenization, and removal of stopwords. Explain the preprocessing techniques
used and their impact on model performance.

In [16]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [17]:
df=pd.read_csv("/content/drive/MyDrive/nlp_dataset.csv")
df

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear
...,...,...
5932,i begun to feel distressed for you,fear
5933,i left feeling annoyed and angry thinking that...,anger
5934,i were to ever get married i d have everything...,joy
5935,i feel reluctant in applying there because i w...,fear


In [18]:
df.head(5)

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


Tokenization

In [19]:
df['Tokens'] = df['Comment'].apply(
    lambda text: str(text).lower().split()
)

display(df[['Comment', 'Tokens']].head())

,Comment,Tokens
0,i seriously hate one subject to death but now ...,"[i, seriously, hate, one, subject, to, death, ..."
1,im so full of life i feel appalled,"[im, so, full, of, life, i, feel, appalled]"
2,i sit here to write i start to dig out my feel...,"[i, sit, here, to, write, i, start, to, dig, o..."
3,ive been really angry with r and i feel like a...,"[ive, been, really, angry, with, r, and, i, fe..."
4,i feel suspicious if there is no one outside l...,"[i, feel, suspicious, if, there, is, no, one, ..."


Explanation: Tokenization splits each comment into individual words called tokens.

Download the stopwords list.

In [20]:
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# Preserve words that can reverse meaning
stop_words = stop_words - {'no', 'not', 'nor'}

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [21]:
stop_words = set(stopwords.words('english'))

# Preserve important negation words
stop_words = stop_words - {'no', 'not', 'nor'}

print(stop_words)

{'will', 'as', 'during', 'any', 'just', 'has', 'this', 'herself', 'he', 'whom', "you'd", 'here', "we've", 'be', 'from', 'most', 'my', 'a', 'can', 're', 'aren', 'him', 'll', "they'll", 'weren', 'when', "aren't", "you're", "we'd", 'of', 'don', 'before', 'until', 'than', 'above', 'you', 'been', "mustn't", 'an', 'only', 'or', 'about', 'we', "don't", "didn't", 'it', 'more', 'wasn', 'few', 'out', "should've", "doesn't", 'being', 'how', 'hers', 'if', 'should', 'is', 'through', 'then', 'to', 'in', 'both', 't', 'that', 'his', 'hasn', 'those', 'needn', "they're", "weren't", 'shouldn', 'the', 'too', 'off', 'was', "haven't", 'into', 'for', 'm', 'all', 'myself', 'what', 'were', 'once', 'our', 'under', 'do', 'himself', 'with', 'd', "they've", 'again', 'there', "wouldn't", "it'll", "i'd", "we're", 'while', 'own', 'hadn', 'won', "won't", 'on', 'because', "they'd", 'but', 'isn', 'yourself', 'below', 'so', 'does', 'which', 'me', 'who', "mightn't", 'very', 'and', "it's", 'did', "hadn't", 'other', "she'll

Explanation: Stopwords are common words such as the, is, and and. We remove them to reduce unnecessary words while preserving important negation words.

Clean the text

In [22]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

In [23]:
df = df.dropna(subset=['Comment', 'Emotion'])

df['Clean_Text'] = df['Comment'].apply(clean_text)

display(df[['Comment', 'Clean_Text', 'Emotion']].head())

,Comment,Clean_Text,Emotion
0,i seriously hate one subject to death but now ...,seriously hate one subject death feel reluctan...,fear
1,im so full of life i feel appalled,im full life feel appalled,anger
2,i sit here to write i start to dig out my feel...,sit write start dig feelings think afraid acce...,fear
3,ive been really angry with r and i feel like a...,ive really angry r feel like idiot trusting fi...,joy
4,i feel suspicious if there is no one outside l...,feel suspicious no one outside like rapture ha...,fear


Explanation: This removes punctuation and stopwords from the comments and creates cleaned text for feature extraction.

2. Feature Extraction

Implement feature extraction using CountVectorizer or TfidfVectorizer. Describe how the
chosen method transforms the text data into numerical features.

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Input and output
X = df['Clean_Text']
y = df['Emotion']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert text into numerical features
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (4749, 7738)
Testing data shape: (1188, 7738)


Explanation: TF-IDF (Term Frequency–Inverse Document Frequency) was used to convert the cleaned text comments into numerical features. It measures the importance of words in a document relative to the entire collection of documents. Words that are more informative receive higher weights, while words that appear frequently across many documents receive lower weights.


The dataset was divided into 80% training data and 20% testing data. The TF-IDF vectorizer was fitted on the training data and then used to transform both datasets. The resulting numerical features were used as input for the Naive Bayes and Support Vector Machine models.

3. Model Development

Train the following machine learning models
a)Naive Bayes b)Support Vector Machine

In [25]:
from sklearn.naive_bayes import MultinomialNB

# Create the model
nb_model = MultinomialNB()

# Train the model
nb_model.fit(X_train, y_train)

# Predict emotions for test comments
nb_predictions = nb_model.predict(X_test)

print("Naive Bayes model trained successfully!")

Naive Bayes model trained successfully!


In [26]:
from sklearn.svm import LinearSVC

# Create the model
svm_model = LinearSVC()

# Train the model
svm_model.fit(X_train, y_train)

# Predict emotions
svm_predictions = svm_model.predict(X_test)

print("SVM model trained successfully!")

SVM model trained successfully!


4. Model Comparison

Evaluate the model using appropriate metrics (e.g., accuracy, F1-score). Provide a brief
explanation of the chosen model and its suitability for emotion classification.

In [27]:
from sklearn.metrics import accuracy_score, f1_score

# Naive Bayes evaluation
nb_accuracy = accuracy_score(y_test, nb_predictions)
nb_f1 = f1_score(y_test, nb_predictions, average='weighted')

# SVM evaluation
svm_accuracy = accuracy_score(y_test, svm_predictions)
svm_f1 = f1_score(y_test, svm_predictions, average='weighted')

# Display results
results = pd.DataFrame({
    'Model': ['Naive Bayes', 'SVM'],
    'Accuracy': [nb_accuracy, svm_accuracy],
    'F1-score': [nb_f1, svm_f1]
})

display(results)

,Model,Accuracy,F1-score
0,Naive Bayes,0.890572,0.890565
1,SVM,0.940236,0.940206


In [28]:
from sklearn.metrics import classification_report

print("Naive Bayes Report:")
print(classification_report(y_test, nb_predictions))

print("SVM Report:")
print(classification_report(y_test, svm_predictions))

Naive Bayes Report:
              precision    recall  f1-score   support

       anger       0.88      0.91      0.89       400
        fear       0.87      0.90      0.89       388
         joy       0.92      0.86      0.89       400

    accuracy                           0.89      1188
   macro avg       0.89      0.89      0.89      1188
weighted avg       0.89      0.89      0.89      1188

SVM Report:
              precision    recall  f1-score   support

       anger       0.95      0.93      0.94       400
        fear       0.94      0.93      0.94       388
         joy       0.93      0.96      0.94       400

    accuracy                           0.94      1188
   macro avg       0.94      0.94      0.94      1188
weighted avg       0.94      0.94      0.94      1188



Explanation: Naive Bayes achieved an accuracy of 89.06% and a weighted F1-score of 89.06%. SVM achieved an accuracy of 94.02% and a weighted F1-score of 94.02%.

The classification report showed that SVM performed better across all three emotion categories: anger, fear, and joy. In particular, SVM achieved a recall of 96% for joy. Therefore, SVM was selected as the better-performing model for this dataset.

Naive Bayes is computationally efficient and suitable as a baseline model, while SVM is effective for text classification because it can learn decision boundaries from high-dimensional TF-IDF features.

Overall, the results demonstrate that TF-IDF combined with SVM can effectively classify emotions in text comments.